In [131]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
import pickle

import pandas as pd
import category_encoders as ce
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import OneHotEncoder
from common import correlation_based_feature_selection as cbfs

In [132]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [133]:
# Read in data
clinical_features = pd.read_pickle("./data/clinical_features.pkl")
print(clinical_features.shape)
Resnet_features = pd.read_pickle("./data/Resnet_features.pkl")
print(Resnet_features.shape)

(360, 10)
(360, 513)


# 1. Input Features

## 1.1 Clinical Features

In [134]:
X_clinical = clinical_features.copy()
X_clinical = X_clinical.drop(
    columns=["Progression Free Survival", "Event"]
)

# apply label encoding
X_clinical = X_clinical.replace(
    to_replace={
        "Sex": {
            "Female": 0,
            "Male": 1,
        },
        "Extent of Tumor Resection": {
            "Not Applicable": 0,
            "Unavailable": 0,
            "Biopsy only": 1,
            "Partial resection": 2,
            "Gross/Near total resection": 3,
            
        },
        "Chemotherapy": {
            "Yes": 1,
            "No": 0,
            "Not Applicable": 0,
            "Not Reported": 0,
            "Unavailable": 0,
        },
        "Radiation": {
            "Yes": 1,
            "No": 0,
            "Not Applicable": 0,
            "Not Reported": 0,
            "Unavailable": 0,
        },
    }
)


X_clinical["NF1"] = X_clinical["NF1"].apply(
    lambda x: 1 if x == "Neurofibromatosis, Type 1 (NF-1)" else 0
)

# split subjects into Discovery and Replicate cohorts
discovery_clinical = X_clinical[X_clinical["Cohort"] == "Discovery"].copy()
replicate_clinical = X_clinical[X_clinical["Cohort"] == "Replicate"].copy()


# apply count encoding
encoder = ce.CountEncoder()
columns = [
    "Tumor Location",
]
encoder.fit(discovery_clinical[columns])
discovery_clinical[columns] = encoder.transform(discovery_clinical[columns])
replicate_clinical[columns] = encoder.transform(replicate_clinical[columns])
with open("./encoder.pkl", mode="wb") as file:
    pickle.dump(encoder, file)

# apply standardization
scaler = StandardScaler()
columns = [
    "Age at Diagnosis",
    "Tumor Location",
]
scaler.fit(discovery_clinical[columns])
discovery_clinical[columns] = scaler.transform(discovery_clinical[columns])
replicate_clinical[columns] = scaler.transform(replicate_clinical[columns])
with open("./scaler_clinical.pkl", mode="wb") as file:
    pickle.dump(scaler, file)

# apply normalization
normalizer = 3
discovery_clinical["Extent of Tumor Resection"] = discovery_clinical[
    "Extent of Tumor Resection"
].apply(lambda x: x / normalizer)
replicate_clinical["Extent of Tumor Resection"] = replicate_clinical[
    "Extent of Tumor Resection"
].apply(lambda x: x / normalizer)

# concatenate cohorts
X_clinical = pd.concat([discovery_clinical, replicate_clinical])

# cache clinical input features
X_clinical.to_pickle("X_clinical.pkl")
X_clinical.shape

(360, 8)

In [135]:
X_clinical

,Sex,Age at Diagnosis,Tumor Location,NF1,Extent of Tumor Resection,Chemotherapy,Radiation,Cohort
SubjectID,,,,,,,,
C1003557,1,-0.531078,1.170136,0,0.333333,1,0,Discovery
C1026189,0,-0.754608,-0.706969,0,0.333333,1,0,Discovery
C1032462,1,-1.139226,-1.547464,0,1.000000,0,0,Discovery
C1032585,1,-0.126520,-1.463415,0,1.000000,0,0,Discovery
C1042056,1,-0.542097,-0.706969,0,0.000000,0,0,Discovery
...,...,...,...,...,...,...,...,...
C829512,0,0.838961,1.170136,0,1.000000,0,0,Replicate
C831849,1,-1.039529,-0.034574,0,1.000000,0,0,Replicate
C868872,0,-0.119174,1.170136,0,1.000000,0,0,Replicate


## 1.2 Radiomic Features

In [136]:
X_Resnet = Resnet_features.copy()
X_Resnet = X_Resnet.drop(columns=["Session"])
# split subjects into Discovery and Replicate cohorts
discovery_resnet = X_Resnet[X_Resnet.index.isin(discovery_clinical.index)].copy()
replicate_resnet = X_Resnet[X_Resnet.index.isin(replicate_clinical.index)].copy()

# apply imputing
imputer = SimpleImputer()
imputer.fit(discovery_resnet)
discovery_resnet.loc[:, :] = imputer.transform(discovery_resnet)
replicate_resnet.loc[:, :] = imputer.transform(replicate_resnet)
with open("./imputer.pkl", mode="wb") as file:
    pickle.dump(imputer, file)

# apply standardization
scaler = StandardScaler()
scaler.fit(discovery_resnet)
discovery_resnet.loc[:, :] = scaler.transform(discovery_resnet)
replicate_resnet.loc[:, :] = scaler.transform(replicate_resnet)
with open("./scaler_radiomic.pkl", mode="wb") as file:
    pickle.dump(scaler, file)

# remove constant features
remover = VarianceThreshold(threshold=0)
remover.fit(discovery_resnet)
column_indices = list(remover.get_support(indices=True))
discovery_resnet = discovery_resnet[discovery_resnet.columns[column_indices]]
replicate_resnet = replicate_resnet[replicate_resnet.columns[column_indices]]
with open("./remover.pkl", mode="wb") as file:
    pickle.dump(remover, file)

# remove correlated features (|r| > 0.9)
dropper = cbfs.main(discovery_resnet)
discovery_resnet = discovery_resnet.drop(columns=dropper)
replicate_resnet = replicate_resnet.drop(columns=dropper)
with open("./dropper.pkl", mode="wb") as file:
    pickle.dump(dropper, file)

# concatenate cohorts
X_resnet = pd.concat([discovery_resnet, replicate_resnet]).copy()
X_resnet["Cohort"] = X_resnet.index.map(
    lambda x: "Discovery" if x in discovery_clinical.index else "Replicate"
)

# cache radiomic features
X_resnet.to_pickle("X_resnet.pkl")
X_resnet.shape

⌛ Time elapsed: 00:00:01


(360, 152)

In [137]:
X_resnet

,feature_015,feature_017,feature_024,feature_035,feature_036,feature_046,feature_061,feature_065,feature_070,feature_071,...,feature_496,feature_497,feature_499,feature_500,feature_505,feature_507,feature_509,feature_510,feature_511,Cohort
SubjectID,,,,,,,,,,,,,,,,,,,,,
C1003557,-1.247891,-0.702801,-1.152636,-1.165135,-0.785172,-0.481135,0.578802,0.148201,-0.485155,-1.016779,...,-0.332478,-0.046607,-0.544620,0.633566,-0.196353,-0.597890,-0.052684,1.047395,-0.821582,Discovery
C1026189,-0.610715,-0.723740,-1.699640,0.056966,-0.744825,-1.045271,-0.084552,-0.596103,-0.045294,-0.922678,...,-0.014157,0.087654,0.893871,1.524050,0.260514,-0.410447,0.741726,0.948676,1.913924,Discovery
C1032462,-0.345129,0.655327,1.287634,1.306441,0.461093,-0.073907,-1.000832,0.340514,1.678334,1.914242,...,0.824044,1.864150,0.647703,-0.246631,0.560116,1.496565,-0.600794,-0.583435,0.721758,Discovery
C1032585,-1.101889,-1.818862,-1.377176,0.353435,-0.377543,-1.425351,-0.785421,-0.189007,0.343328,-1.371552,...,1.688842,0.409977,1.437075,0.880203,0.691147,0.851352,2.285647,1.056225,0.867008,Discovery
C1042056,0.343271,-0.448257,0.242800,-0.885028,0.468232,1.684497,0.602148,-1.099826,-0.989288,-0.749312,...,-1.622206,-0.772091,-1.204866,-0.496641,-1.051254,-0.983642,-0.559908,-0.527506,-0.666214,Discovery
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
C829512,-0.096093,1.104075,0.645006,1.479944,-0.209483,-0.835733,-1.464012,1.287286,0.285518,0.671785,...,1.065697,0.446245,0.636761,-0.747655,0.110031,0.302011,-0.625069,-0.221615,-0.498393,Replicate
C831849,0.472054,0.022834,1.703089,-1.126579,1.555182,0.248930,0.099408,0.719875,0.482973,0.616916,...,0.211443,-0.376090,-0.841109,-1.025591,0.380121,1.371347,-0.306069,-1.290730,0.049521,Replicate
C868872,-0.976514,-1.964505,-1.993158,-0.784514,-1.472197,-1.407104,1.652783,0.364791,0.646490,-1.887362,...,0.003312,0.392909,-0.348585,2.283065,0.474332,-0.014402,0.592916,1.017834,1.468718,Replicate


# 2. Output Features

In [138]:
# select columns
y = clinical_features[["Progression Free Survival", "Event"]].merge(
    X_clinical["Cohort"].to_frame(), left_index=True, right_index=True
)
# compute age in months
y["Progression Free Survival"] = y["Progression Free Survival"].apply(
    lambda x: int(x) / 30.417
)

# cache output features
y.to_pickle("./y.pkl")
y.shape

(360, 3)